In [5]:
import os
import operator
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.runnables import RunnableConfig

from typing_extensions import TypedDict
from typing import List, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# from langgraph.runtime import Runtime
# from langgraph.checkpoint.memory import InMemorySaver # In memory checkpointing for graph -> short term memory
# from langgraph.store.memory import InMemoryStore # Long term storage for graph -> long term memory

In [9]:
class OverallState(TypedDict): # This state will be passed as the base state of the graph
    card_id: str
    raw_items: List[dict]
    final_receipt: str
    processed_items: Annotated[List[str], operator.add] # Here reducer does concatenation of list of strings
    total_amount: Annotated[float, operator.add] # Here reducer does addition of floats


class CartItemWorkerState(TypedDict): # This state will be passed as the private state to fanout worker nodes for map step
    item_name: str
    price: float

In [ ]:
def fetch_cart_items(state: OverallState):
    items_from_db=[
        {"item_name": "Apple macbook pro", "price": 1000},
        {"item_name": "Apple macbook air", "price": 800},
        {"item_name": "Apple macbook mini", "price": 600},
    ]

    return {
        "raw_items": items_from_db
    }

In [10]:
def fanout_calculate_item_total(state: CartItemWorkerState):
    name=state["item_name"]
    price=state["price"]

    final_price=(price*0.9)*1.05
    
    summary=f"{name}: ${final_price} (discounted by 10% and taxed 5%)"

    return {
        "processed_items": [summary],
        "total_amount": final_price,
    }

In [11]:
def generate_receipt(state: OverallState):
    lines=["="*40, f"RECEIPT FOR CART #{state["card_id"]}", "="*40]

    for index, line in enumerate(state["processed_items"], start=1):
        lines.append(f"{index}. {line}")

    lines.append("="*40)

    lines.append(f"Total Amount: ${state["total_amount"]}")

    return {
        "final_receipt": "\n".join(lines)
    }

In [ ]:
def map_items_to_fanout_nodes(state: OverallState): # This is not a node, but a function to put routing logic in a conditional edge
    return [
        Send( # Send takes two inputs
            node="fanout_calculate_item_total", # 1 is the node which will act as the fanout node
            arg={"item_name": items["item_name"], "price": items["price"]} # 2 is the private state input to fanout node
        )
        for items in state["raw_items"]
    ]

In [15]:
builder=StateGraph(state_schema=OverallState)

builder.add_node("fetch_cart_items", fetch_cart_items)
builder.add_node("fanout_calculate_item_total", fanout_calculate_item_total)
builder.add_node("generate_receipt", generate_receipt)

builder.add_edge(START, "fetch_cart_items")
builder.add_conditional_edges("fetch_cart_items", map_items_to_fanout_nodes) # This will trigger all fanout_calculate_item_total nodes in parallel
builder.add_edge("fanout_calculate_item_total", "generate_receipt")
builder.add_edge("generate_receipt", END)

graph=builder.compile()

In [16]:
initial_state={
    "card_id": "1234567890",
    "raw_items": [],
    "final_receipt": "",
    "processed_items": [],
    "total_amount": 0
}

In [17]:
result=graph.invoke(initial_state)
print(result)

{'card_id': '1234567890', 'raw_items': [{'item_name': 'Apple macbook pro', 'price': 1000}, {'item_name': 'Apple macbook air', 'price': 800}, {'item_name': 'Apple macbook mini', 'price': 600}], 'final_receipt': '========================================\nRECEIPT FOR CART #1234567890\n========================================\n1. Apple macbook pro: $945.0 (discounted by 10% and taxed 5%)\n2. Apple macbook air: $756.0 (discounted by 10% and taxed 5%)\n3. Apple macbook mini: $567.0 (discounted by 10% and taxed 5%)\n========================================\nTotal Amount: $2268.0', 'processed_items': ['Apple macbook pro: $945.0 (discounted by 10% and taxed 5%)', 'Apple macbook air: $756.0 (discounted by 10% and taxed 5%)', 'Apple macbook mini: $567.0 (discounted by 10% and taxed 5%)'], 'total_amount': 2268.0}


In [18]:
print(result["final_receipt"])

RECEIPT FOR CART #1234567890
1. Apple macbook pro: $945.0 (discounted by 10% and taxed 5%)
2. Apple macbook air: $756.0 (discounted by 10% and taxed 5%)
3. Apple macbook mini: $567.0 (discounted by 10% and taxed 5%)
Total Amount: $2268.0
